<h1>5. Production Patterns</h1>

In [1]:
import warnings

# Suppress Pydantic V2 warnings BEFORE importing CrewAI
warnings.filterwarnings('ignore', category=DeprecationWarning, module='pydantic.*')
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic.*')

import os
import json
import yaml
import litellm
from pathlib import Path
from typing import Dict, List, Any, Callable
from dotenv import load_dotenv
import logging

from collections import Counter
from pprint import pprint

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

from functools import wraps
import time

from datetime import datetime

from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool

import watermark

%load_ext watermark
%matplotlib inline

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/IPython/core/pylabtools.py:77: DeprecationWarning: backend2gui is deprecated since IPython 8.24, backends are managed in matplotlib and can be externally registered.
  warnings.warn(


We start by printing out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.10.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.2.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 2b98c2c19a22d9b5e74349315b8e0b33a6a32a32

crewai      : 0.86.0
json        : 2.0.9
pandas      : 2.2.3
yaml        : 6.0.1
crewai_tools: 0.17.0
numpy       : 1.26.4
matplotlib  : 3.8.4
watermark   : 2.5.0
litellm     : 1.60.0
logging     : 0.5.1.2



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')

# 5. Production Patterns

In this notebook, we'll cover essential patterns for production-ready agent systems:
- **Extensible configurations**: Separating agent definitions from code
- **Multi-LLM support**: Using different models for different agents
- **Error handling**: Graceful failures and recovery
- **Logging and monitoring**: Observability for agent systems
- **Testing strategies**: How to test multi-agent systems
- **Best practices**: Lessons learned from production deployments

These patterns transform proof-of-concepts into maintainable, reliable production systems.

## Setup

In [4]:
# Configure LiteLLM to handle Anthropic API message format requirements
litellm.modify_params = True

load_dotenv()

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## Pattern 1: Configuration-Driven Agents

Hardcoding agent definitions makes systems brittle. Instead, define agents in configuration files that can be:
- Modified without code changes
- Version controlled separately
- Loaded dynamically
- Shared across teams

In [5]:
# Example agent configuration (YAML)
agent_config_yaml = """
agents:
  researcher:
    role: "Senior Market Researcher"
    goal: "Conduct thorough market research and competitive analysis"
    backstory: |
      You are a senior market researcher with 15 years of experience.
      You specialize in technology markets and have a track record of
      identifying emerging trends before competitors.
    tools:
      - search
    llm_model: "claude-sonnet-4-5-20250929"
    temperature: 0.3
    verbose: true
    allow_delegation: false
  
  analyst:
    role: "Data Analyst"
    goal: "Analyze data and provide actionable insights"
    backstory: |
      You are a data analyst who excels at finding patterns
      and translating data into business recommendations.
    tools: []
    llm_model: "claude-sonnet-4-5-20250929"
    temperature: 0.1
    verbose: true
    allow_delegation: false
  
  writer:
    role: "Content Writer"
    goal: "Create engaging, well-structured content"
    backstory: |
      You are a professional writer who creates clear,
      compelling content for business audiences.
    tools: []
    llm_model: "claude-haiku-4-5-20251001"  # Faster model for writing
    temperature: 0.7
    verbose: true
    allow_delegation: false

tasks:
  research:
    description: "Research {topic} and provide key findings"
    expected_output: "Research summary with sources"
    agent: "researcher"
  
  analyze:
    description: "Analyze research findings and identify top 3 insights"
    expected_output: "Analysis with actionable insights"
    agent: "analyst"
    context: ["research"]
  
  write:
    description: "Write a 500-word summary report"
    expected_output: "Executive summary report"
    agent: "writer"
    context: ["research", "analyze"]
"""

Save config to file

In [6]:
os.makedirs('./config', exist_ok=True)
with open('./config/agents.yaml', 'w') as f:
    f.write(agent_config_yaml)

print("Agent configuration saved to ./config/agents.yaml")

Agent configuration saved to ./config/agents.yaml


## Pattern 2: Agent Factory

Create a factory class to build agents from configuration.

In [7]:
class AgentFactory:
    """Factory for creating agents from configuration."""
    
    def __init__(self, config_path: str):
        """Initialize with path to YAML config file."""
        with open(config_path, 'r') as f:
            self.config = yaml.safe_load(f)
        
        # Tool registry
        self.tools = {
            'search': SerperDevTool()
        }
        
        logger.info(f"Loaded config from {config_path}")
    
    def _get_llm(self, model: str, temperature: float) -> LLM:
        """Get LLM instance with specified parameters."""
        # Add anthropic/ prefix for LiteLLM
        if not model.startswith('anthropic/'):
            model = f"anthropic/{model}"
        return LLM(model=model, temperature=temperature)
    
    def create_agent(self, agent_name: str) -> Agent:
        """Create an agent from config."""
        if agent_name not in self.config['agents']:
            raise ValueError(f"Agent {agent_name} not found in config")
        
        agent_config = self.config['agents'][agent_name]
        
        # Get tools
        tools = [self.tools[t] for t in agent_config.get('tools', [])]
        
        # Get LLM
        llm = self._get_llm(
            agent_config.get('llm_model', 'claude-sonnet-4-5-20250929'),
            agent_config.get('temperature', 0.7)
        )
        
        agent = Agent(
            role=agent_config['role'],
            goal=agent_config['goal'],
            backstory=agent_config['backstory'],
            tools=tools,
            llm=llm,
            verbose=agent_config.get('verbose', True),
            allow_delegation=agent_config.get('allow_delegation', False)
        )
        
        logger.info(f"Created agent: {agent_name} ({agent_config['role']})")
        return agent
    
    def create_all_agents(self) -> Dict[str, Agent]:
        """Create all agents defined in config."""
        return {
            name: self.create_agent(name)
            for name in self.config['agents'].keys()
        }
    
    def create_task(self, task_name: str, agents: Dict[str, Agent], 
                   tasks: Dict[str, Task] = None, **kwargs) -> Task:
        """Create a task from config."""
        if task_name not in self.config['tasks']:
            raise ValueError(f"Task {task_name} not found in config")
        
        task_config = self.config['tasks'][task_name]
        
        # Format description with kwargs
        description = task_config['description'].format(**kwargs)
        
        # Get context tasks
        context = None
        if 'context' in task_config and tasks:
            context = [tasks[t] for t in task_config['context']]
        
        task = Task(
            description=description,
            expected_output=task_config['expected_output'],
            agent=agents[task_config['agent']],
            context=context
        )
        
        logger.info(f"Created task: {task_name}")
        return task

Test the factory

In [8]:
factory = AgentFactory('./config/agents.yaml')
agents = factory.create_all_agents()

print("\nCreated agents:")
for name, agent in agents.items():
    print(f"  - {name}: {agent.role}")


Created agents:
  - researcher: Senior Market Researcher
  - analyst: Data Analyst
  - writer: Content Writer


## Pattern 3: Error Handling and Retries

Production systems need robust error handling, so we define a retry decorator

In [9]:
def retry_with_backoff(max_retries: int = 3, base_delay: float = 1.0):
    """Decorator to retry function with exponential backoff."""
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs):
            retries = 0
            while retries < max_retries:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    retries += 1
                    if retries >= max_retries:
                        logger.error(f"Failed after {max_retries} retries: {str(e)}")
                        raise
                    
                    delay = base_delay * (2 ** (retries - 1))
                    logger.warning(
                        f"Attempt {retries} failed: {str(e)}. "
                        f"Retrying in {delay}s..."
                    )
                    time.sleep(delay)
        return wrapper
    return decorator

Define an Executor class that uses the retry decorator

In [10]:
class SafeCrewExecutor:
    """Safe execution wrapper for CrewAI crews."""
    
    def __init__(self, crew: Crew):
        self.crew = crew
    
    @retry_with_backoff(max_retries=3, base_delay=2.0)
    def execute(self, inputs: Dict[str, Any] = None) -> str:
        """Execute crew with retry logic."""
        try:
            logger.info("Starting crew execution")
            result = self.crew.kickoff(inputs=inputs or {})
            logger.info("Crew execution completed successfully")
            return result
        except Exception as e:
            logger.error(f"Crew execution failed: {str(e)}")
            raise
    
    def execute_with_fallback(self, inputs: Dict[str, Any] = None, 
                             fallback_fn: Callable = None) -> str:
        """Execute with fallback function if crew fails."""
        try:
            return self.execute(inputs)
        except Exception as e:
            logger.error(f"All retries failed: {str(e)}")
            if fallback_fn:
                logger.info("Executing fallback function")
                return fallback_fn(inputs, e)
            raise

print("Safe executor pattern defined")

Safe executor pattern defined


#### Pattern 4: Multi-LLM Strategy

Different agents can use different LLMs based on:
- Cost (use GPT-4 for simple tasks, GPT-5 for complex)
- Speed (faster models for real-time responses)
- Capability (specialized models for specific domains)

In [11]:
class LLMSelector:
    """Select appropriate LLM based on task requirements."""
    
    LLM_PROFILES = {
        'fast': {
            'model': 'anthropic/claude-haiku-4-5-20251001',
            'temperature': 0.7,
            'use_case': 'Simple tasks, drafts, quick responses'
        },
        'balanced': {
            'model': 'anthropic/claude-sonnet-4-5-20250929',
            'temperature': 0.5,
            'use_case': 'General purpose, good quality/cost balance'
        },
        'precise': {
            'model': 'anthropic/claude-sonnet-4-5-20250929',
            'temperature': 0.1,
            'use_case': 'Analysis, data processing, factual tasks'
        },
        'creative': {
            'model': 'anthropic/claude-sonnet-4-5-20250929',
            'temperature': 0.9,
            'use_case': 'Creative writing, brainstorming'
        }
    }
    
    @classmethod
    def get_llm(cls, profile: str = 'balanced') -> LLM:
        """Get LLM for specified profile."""
        if profile not in cls.LLM_PROFILES:
            raise ValueError(f"Unknown profile: {profile}")
        
        config = cls.LLM_PROFILES[profile]
        logger.info(f"Selected LLM profile: {profile} - {config['use_case']}")
        
        return LLM(
            model=config['model'],
            temperature=config['temperature']
        )
    
    @classmethod
    def recommend_profile(cls, task_type: str) -> str:
        """Recommend LLM profile based on task type."""
        recommendations = {
            'research': 'balanced',
            'analysis': 'precise',
            'writing': 'creative',
            'data_processing': 'precise',
            'chat': 'fast',
            'code': 'precise'
        }
        return recommendations.get(task_type, 'balanced')

Example usage

In [12]:
print("LLM Profiles:")
for profile, config in LLMSelector.LLM_PROFILES.items():
    print(f"  {profile}: {config['model']} (temp={config['temperature']})")
    print(f"    Use case: {config['use_case']}")

LLM Profiles:
  fast: anthropic/claude-haiku-4-5-20251001 (temp=0.7)
    Use case: Simple tasks, drafts, quick responses
  balanced: anthropic/claude-sonnet-4-5-20250929 (temp=0.5)
    Use case: General purpose, good quality/cost balance
  precise: anthropic/claude-sonnet-4-5-20250929 (temp=0.1)
    Use case: Analysis, data processing, factual tasks
  creative: anthropic/claude-sonnet-4-5-20250929 (temp=0.9)
    Use case: Creative writing, brainstorming


Create agents with different LLM profiles

In [14]:
fast_agent = Agent(
    role="Quick Responder",
    goal="Provide fast responses",
    backstory="You provide quick, helpful responses.",
    llm=LLMSelector.get_llm('fast')
)

precise_agent = Agent(
    role="Data Analyst",
    goal="Provide accurate analysis",
    backstory="You perform precise data analysis.",
    llm=LLMSelector.get_llm('precise')
)

print("\nCreated agents with optimized LLM profiles")


Created agents with optimized LLM profiles


## Pattern 5: Comprehensive Logging and Monitoring

Production systems need observability.

In [15]:
class CrewMonitor:
    """Monitor and log crew execution metrics."""
    
    def __init__(self, log_dir: str = './logs'):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(exist_ok=True)
        self.execution_logs: List[Dict] = []
    
    def start_execution(self, crew_name: str, inputs: Dict) -> str:
        """Log execution start."""
        execution_id = f"{crew_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        log_entry = {
            'execution_id': execution_id,
            'crew_name': crew_name,
            'start_time': datetime.now().isoformat(),
            'inputs': inputs,
            'status': 'started'
        }
        
        self.execution_logs.append(log_entry)
        logger.info(f"Started execution: {execution_id}")
        return execution_id
    
    def end_execution(self, execution_id: str, result,
                     error: Exception = None):
        """Log execution completion."""
        for log in self.execution_logs:
            if log['execution_id'] == execution_id:
                log['end_time'] = datetime.now().isoformat()
                log['status'] = 'failed' if error else 'completed'
                log['result'] = str(result) if not error else None
                log['error'] = str(error) if error else None
                
                # Calculate duration
                start = datetime.fromisoformat(log['start_time'])
                end = datetime.fromisoformat(log['end_time'])
                log['duration_seconds'] = (end - start).total_seconds()
                
                # Save to file
                log_file = self.log_dir / f"{execution_id}.json"
                with open(log_file, 'w') as f:
                    json.dump(log, f, indent=2)
                
                logger.info(
                    f"Completed execution: {execution_id} "
                    f"({log['duration_seconds']:.2f}s) - {log['status']}"
                )
                break
    
    def get_metrics(self) -> Dict:
        """Get execution metrics."""
        completed = [l for l in self.execution_logs if l['status'] == 'completed']
        failed = [l for l in self.execution_logs if l['status'] == 'failed']
        
        if completed:
            avg_duration = sum(l['duration_seconds'] for l in completed) / len(completed)
        else:
            avg_duration = 0
        
        return {
            'total_executions': len(self.execution_logs),
            'completed': len(completed),
            'failed': len(failed),
            'success_rate': len(completed) / len(self.execution_logs) if self.execution_logs else 0,
            'avg_duration_seconds': avg_duration
        }

Example usage

In [16]:
monitor = CrewMonitor()

Simulate an execution

In [17]:
exec_id = monitor.start_execution(
    'research_crew',
    {'topic': 'AI trends'}
)

# Simulate work
time.sleep(0.1)

monitor.end_execution(exec_id, "Research completed successfully")

print("\nExecution Metrics:")
pprint(monitor.get_metrics())


Execution Metrics:
{'avg_duration_seconds': 0.105345,
 'completed': 1,
 'failed': 0,
 'success_rate': 1.0,
 'total_executions': 1}


## Putting It All Together: Production-Ready System

Let's combine all patterns into a complete production system.

In [18]:
class ProductionCrewSystem:
    """Complete production-ready crew system."""
    
    def __init__(self, config_path: str, log_dir: str = './logs'):
        self.factory = AgentFactory(config_path)
        self.monitor = CrewMonitor(log_dir)
        self.agents = self.factory.create_all_agents()
        logger.info("Production crew system initialized")
    
    def create_crew(self, crew_name: str, task_names: List[str], 
                   process: Process = Process.sequential) -> Crew:
        """Create a crew from configuration."""
        tasks = {}
        task_list = []
        
        for task_name in task_names:
            task = self.factory.create_task(
                task_name,
                self.agents,
                tasks,
                topic="AI automation"  # Default topic
            )
            tasks[task_name] = task
            task_list.append(task)
        
        crew = Crew(
            agents=list(self.agents.values()),
            tasks=task_list,
            process=process,
            verbose=True
        )
        
        return crew
    
    def execute(self, crew: Crew, crew_name: str, 
               inputs: Dict = None) -> str:
        """Execute crew with full monitoring and error handling."""
        inputs = inputs or {}
        execution_id = self.monitor.start_execution(crew_name, inputs)
        
        try:
            executor = SafeCrewExecutor(crew)
            result = executor.execute(inputs)
            self.monitor.end_execution(execution_id, result)
            return result
        
        except Exception as e:
            self.monitor.end_execution(execution_id, "", error=e)
            raise
    
    def get_system_health(self) -> Dict:
        """Get system health metrics."""
        metrics = self.monitor.get_metrics()
        
        health = {
            'status': 'healthy' if metrics['success_rate'] > 0.8 else 'degraded',
            'metrics': metrics,
            'agents': {
                name: agent.role 
                for name, agent in self.agents.items()
            }
        }
        
        return health

Create production system

In [19]:
system = ProductionCrewSystem('./config/agents.yaml')

In [20]:
print("\n" + "="*80)
print("Production System Ready")
print("="*80)


Production System Ready


Create and execute a crew

In [21]:
crew = system.create_crew(
    'market_research',
    ['research', 'analyze', 'write']
)

result = system.execute(
    crew,
    'market_research',
    {'topic': 'AI automation in healthcare'}
)

print("\n" + "="*80)
print("RESULT:")
print("="*80)
print(result)

# Agent: Senior Market Researcher
## Task: Research AI automation and provide key findings


/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Senior Market Researcher
## Thought: I need to conduct thorough research on AI automation to provide comprehensive findings with sources. Let me start by searching for current information on AI automation trends, applications, and market insights.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"AI automation trends 2024 market research statistics\"}"
## Tool Output: 

Search results: Title: AI & Automation Trends: 2024 Insights & 2025 Outlook
Link: https://info.aiim.org/aiim-blog/ai-automation-trends-2024-insights-2025-outlook
Snippet: Read about the top IM trends of 2024, including persistent paper-based processes, automation maturity gaps, data quality challenges, RAG, ...
---
Title: The State of AI: Global Survey 2025
Link: https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai
Snippet: Our latest survey shows a larger share of respondents reporting AI use by their organizations, though most have yet to scale the technolog

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Senior Market Researcher
## Thought: Thought: Good start. Now let me search for more specific information about AI automation applications and use cases across different industries.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"AI automation applications industries use cases 2024\"}"
## Tool Output: 

Search results: Title: AI use cases by industry, function and type | Deloitte US
Link: https://www.deloitte.com/us/en/what-we-do/capabilities/applied-artificial-intelligence/content/ai-use-cases.html
Snippet: Explore our regularly updated collection of high-impact AI use cases by industry, sector, enterprise function and type, including agentic AI and beyond.
---
Title: 50+ Gen AI Use Cases for the Top 10 Industries in 2024 - Lyzr
Link: https://www.lyzr.ai/blog/gen-ai-use-cases-for-top-industries/
Snippet: Small businesses can utilize Gen AI to automate social media posting, respond to customer inquiries, and analyze engagement metrics. This ...
---
T

...



# Agent: Senior Market Researcher
## Thought: Thought: Let me search for information about the competitive landscape and key players in AI automation.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"AI automation market leaders companies competitive landscape 2024\"}"
## Tool Output: 

Search results: Title: Competitor Analysis of the Artificial Intelligence Market 2025
Link: https://finance.yahoo.com/news/competitor-analysis-artificial-intelligence-market-161500385.html
Snippet: The global artificial intelligence (AI) market is projected to soar from an estimated $184.15 billion in 2024 to approximately $2.53 ...
---
Title: The state of AI in early 2024 - McKinsey
Link: https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai-2024
Snippet: As generative AI adoption accelerates, survey respondents report measurable benefits and increased mitigation of the risk of inaccuracy.
---
Title: AI and Automation Trends Reshaping Business in 2024

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Senior Market Researcher
## Thought: Thought: Let me search for information about ROI and business impact of AI automation.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"AI automation ROI business impact productivity gains 2024\"}"
## Tool Output: 

Search results: Title: AI and Automation Trends Reshaping Business in 2024
Link: https://calvettiferguson.com/ai-automation-trends-2024/
Snippet: Executives are focusing on AI to drive growth, streamline processes, increase productivity, and improve client satisfaction. Through intelligent automation ...
---
Title: Does AI Increase Productivity? Unpacking the Impact
Link: https://gloat.com/blog/does-ai-increase-productivity/
Snippet: AI can increase productivity by automating repetitive tasks, accelerating decision-making, and enabling employees to complete work faster ...
---
Title: AI in the workplace: A report for 2025
Link: https://www.mckinsey.com/capabilities/tech-and-ai/our-insights/superagency-i

/Users/bgoncalves/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/tools/tool_usage.py:162: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Senior Market Researcher
## Thought: Thought: Let me search for future predictions and emerging trends in AI automation.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"AI automation future trends predictions 2025 emerging technologies\"}"
## Tool Output: 

Search results: Title: AI In 2026: 10 Predictions On Automation And The Future Of Work
Link: https://www.forbes.com/sites/charlestowersclark/2025/12/10/ai-in-2026-10-predictions-on-automation-and-the-future-of-work/
Snippet: 1. Expect An AI Valuation Correction · 2. There Is Unlikely To Be A Bursting Of The AI Investment Bubble. · 3. But AGI Won't Just Come From Bigger ...
---
Title: The State of AI: Global Survey 2025 - McKinsey
Link: https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai
Snippet: In this 2025 edition of the annual McKinsey Global Survey on AI, we look at the current trends that are driving real value from artificial ...
---
Title: Tech Trends 2026 | Delo

Check system health

In [22]:
print("\n" + "="*80)
print("SYSTEM HEALTH:")
print("="*80)
pprint(system.get_system_health())


SYSTEM HEALTH:
{'agents': {'analyst': 'Data Analyst',
            'researcher': 'Senior Market Researcher',
            'writer': 'Content Writer'},
 'metrics': {'avg_duration_seconds': 220.335433,
             'completed': 1,
             'failed': 0,
             'success_rate': 1.0,
             'total_executions': 1},
 'status': 'healthy'}
